In [ ]:
# Cell 0: Setup
import subprocess, os

os.chdir("/kaggle/working")
if not os.path.exists("motion-id"):
    subprocess.run(
        ["git", "clone", "https://github.com/Sreekarji/motion-id.git"],
        check=True)
os.chdir("/kaggle/working/motion-id")

subprocess.run([
    "pip", "install", "-q",
    "torch", "numpy", "scipy", "pandas", "scikit-learn",
    "tqdm", "pytorch_metric_learning"
], check=True)

print("Setup complete.")
print(f"Working dir: {os.getcwd()}")

In [ ]:
# Cell 1: Verify input paths
import os

BASE = "/kaggle/input/datasets/djaarf"

PATHS = {
    "MPI Part 1": f"{BASE}/motionid-imu-all-motions-part1",
    "MPI Part 2": f"{BASE}/motionid-imu-all-motions-part2",
    "MPI Part 3": f"{BASE}/motionid-imu-all-motions-part3",
    "UV":         f"{BASE}/motionid-imu-specific-motion",
}
for name, path in PATHS.items():
    exists = os.path.exists(path)
    print(f"{'OK' if exists else 'X'} {name}: {path}")
    if exists:
        entries = os.listdir(path)
        print(f"    Contents: {entries[:5]}")

In [ ]:
# Cell 2: MPI Preprocessing
import sys, os
sys.path.insert(0, "/kaggle/working/motion-id")
os.chdir("/kaggle/working/motion-id")

from preprocessing.mpi_preprocess import discover_sessions, process_session
import pandas as pd, numpy as np

PROCESSED_MPI = "/kaggle/working/data/mpi/processed"
os.makedirs(PROCESSED_MPI, exist_ok=True)

sessions = discover_sessions([
    "/kaggle/input/datasets/djaarf/motionid-imu-all-motions-part1/IMU_all_motions_part1",
    "/kaggle/input/datasets/djaarf/motionid-imu-all-motions-part2/IMU_all_motions_part2",
    "/kaggle/input/datasets/djaarf/motionid-imu-all-motions-part3/IMU_all_motions_part3",
])
print(f"Found {len(sessions)} MPI sessions")

rows = []
for uid, did, sdir in sessions:
    print(f"  user={uid} device={did}...", end=" ")
    X, y = process_session(uid, did, sdir)
    key  = f"{uid}_{did}"
    if X is None:
        print("N/A")
        rows.append({"user_id": uid, "device_id": did,
                     "n_pos": 0, "n_neg": 0, "status": "N/A"})
    else:
        np.savez(os.path.join(PROCESSED_MPI, f"{key}.npz"), X=X, y=y)
        n_pos, n_neg = int((y==1).sum()), int((y==0).sum())
        print(f"X={X.shape}, pos={n_pos}, neg={n_neg}")
        rows.append({"user_id": uid, "device_id": did,
                     "n_pos": n_pos, "n_neg": n_neg, "status": "OK"})

mf = pd.DataFrame(rows)
mf.to_csv(os.path.join(PROCESSED_MPI, "manifest.csv"), index=False)
print(f"\nMPI done. Valid: {(mf.status=='OK').sum()}/{len(mf)}")

In [ ]:
# Cell 3: UV Preprocessing
import sys, os, numpy as np
sys.path.insert(0, "/kaggle/working/motion-id")
os.chdir("/kaggle/working/motion-id")

from utils.bin_reader import load_session, FLAG_USER_PRESENT
from preprocessing.uv_preprocess import compute_features, extract_window

UV_INPUT_DIR  = "/kaggle/input/datasets/djaarf/motionid-imu-specific-motion/IMU_specific_motion/train_val_test"
PROCESSED_UV  = "/kaggle/working/data/uv/processed"
os.makedirs(PROCESSED_UV, exist_ok=True)

if not os.path.exists(UV_INPUT_DIR):
    print(f"ERROR: {UV_INPUT_DIR} not found")
else:
    user_dirs = sorted(
        d for d in os.listdir(UV_INPUT_DIR)
        if os.path.isdir(os.path.join(UV_INPUT_DIR, d)))
    print(f"Found {len(user_dirs)} UV users")

    for uid in user_dirs:
        base = os.path.join(UV_INPUT_DIR, uid, "s20")
        if not os.path.exists(base):
            print(f"  user={uid}: no s20/  skip"); continue
        sessions = sorted(os.listdir(base))
        if not sessions: continue
        sdir = os.path.join(base, sessions[0])

        merged, screen = load_session(sdir)
        if merged is None:
            print(f"  user={uid}: load failed"); continue

        unlocks = (screen[screen["event"] == FLAG_USER_PRESENT]
                   .sort_values("timestamp"))
        feats, clusters, n_skip = [], [], 0
        for trial_idx, (_, row) in enumerate(unlocks.iterrows()):
            w = extract_window(merged, int(row["timestamp"]))
            if w is None: n_skip += 1; continue
            feats.append(compute_features(w))
            clusters.append(trial_idx // 50)

        if not feats:
            print(f"  user={uid}: no valid trials (skipped {n_skip})"); continue
        F = np.stack(feats, 0).astype(np.float32)
        C = np.array(clusters, dtype=np.int64)
        np.savez(os.path.join(PROCESSED_UV, f"{uid}.npz"),
                 features=F, cluster_ids=C, user_id=uid)
        skip_str = f", skipped {n_skip}" if n_skip else ""
        print(f"  user={uid}: {F.shape}{skip_str}")

    done = len([f for f in os.listdir(PROCESSED_UV) if f.endswith(".npz")])
    print(f"\nUV preprocessing done. {done} users saved.")

In [ ]:
# Cell 4: Shape verification before training
import os, numpy as np

PROCESSED_MPI = "/kaggle/working/data/mpi/processed"
PROCESSED_UV  = "/kaggle/working/data/uv/processed"

print("=== MPI Processed Data ===")
mpi_files = [f for f in os.listdir(PROCESSED_MPI) if f.endswith(".npz")]
print(f"Files: {len(mpi_files)}")
if mpi_files:
    d = np.load(os.path.join(PROCESSED_MPI, mpi_files[0]))
    print(f"Sample: X={d['X'].shape}, y={d['y'].shape}")
    assert d['X'].ndim == 3 and d['X'].shape[1] == 18 and d['X'].shape[2] == 150
    print("Shape assertion passed: (N, 18, 150)")

print("\n=== UV Processed Data ===")
uv_files = [f for f in os.listdir(PROCESSED_UV) if f.endswith(".npz")]
print(f"Files: {len(uv_files)}")
if uv_files:
    d = np.load(os.path.join(PROCESSED_UV, uv_files[0]))
    print(f"Sample: features={d['features'].shape}, cluster_ids={d['cluster_ids'].shape}")
    assert d['features'].ndim == 4
    assert d['features'].shape[1:] == (22, 4, 50)
    print("Shape assertion passed: (N, 22, 4, 50)")

print("\nAll shape checks passed.")

In [ ]:
# Cell 5: MPI Training (resumable)
import sys, os
sys.path.insert(0, "/kaggle/working/motion-id")
os.chdir("/kaggle/working/motion-id")

os.makedirs("/kaggle/working/models/checkpoints", exist_ok=True)
os.makedirs("/kaggle/working/evaluation", exist_ok=True)

import training.train_mpi as train_mpi_mod

train_mpi_mod.PROCESSED_DIR = "/kaggle/working/data/mpi/processed"
train_mpi_mod.CKPT_DIR      = "/kaggle/working/models/checkpoints"
train_mpi_mod.RESULTS_DIR   = "/kaggle/working/evaluation"
train_mpi_mod.RESUME_LOG    = "/kaggle/working/evaluation/mpi_completed.csv"

train_mpi_mod.main(use_dummy=False)

In [ ]:
# Cell 6: UV Baseline Training - n=75 (~18 min on P100)
import sys, os
sys.path.insert(0, "/kaggle/working/motion-id")
os.chdir("/kaggle/working/motion-id")

import training.train_uv as train_uv_mod

train_uv_mod.PROCESSED_DIR = "/kaggle/working/data/uv/processed"
train_uv_mod.CKPT_DIR      = "/kaggle/working/models/checkpoints"
train_uv_mod.RESULTS_DIR   = "/kaggle/working/evaluation"

train_uv_mod.main(
    use_dummy=False,
    n_baseline=75,
    all_splits=False)

In [ ]:
# Cell 7: UV Training - all 6 splits (optional, ~4 hr)
import sys, os
sys.path.insert(0, "/kaggle/working/motion-id")
os.chdir("/kaggle/working/motion-id")

import training.train_uv as train_uv_mod

train_uv_mod.PROCESSED_DIR = "/kaggle/working/data/uv/processed"
train_uv_mod.CKPT_DIR      = "/kaggle/working/models/checkpoints"
train_uv_mod.RESULTS_DIR   = "/kaggle/working/evaluation"

train_uv_mod.main(
    use_dummy=False,
    n_baseline=None,
    all_splits=True)

In [ ]:
# Cell 8: Print paper-style Tables 1, 2, 3
import sys, os
sys.path.insert(0, "/kaggle/working/motion-id")
os.chdir("/kaggle/working/motion-id")

from evaluation.evaluate import print_table1, print_table2, print_table3

RESULTS_DIR = "/kaggle/working/evaluation"

print_table1(os.path.join(RESULTS_DIR, "results_mpi.csv"))
print_table2(os.path.join(RESULTS_DIR, "results_baseline.csv"))
print_table3(os.path.join(RESULTS_DIR, "results_uv_final.csv"))

In [ ]:
# Cell 9: List all saved outputs
import os

OUTPUT_ROOT = "/kaggle/working"
for dirpath, _, filenames in os.walk(OUTPUT_ROOT):
    for fname in filenames:
        fpath = os.path.join(dirpath, fname)
        size  = os.path.getsize(fpath)
        rel   = os.path.relpath(fpath, OUTPUT_ROOT)
        print(f"  {rel:<60} {size/1024:.1f} KB")

print("\nClick Save Version to persist these outputs.")

In [ ]:
# Cell 10: Copy evaluation results to repo and push
import subprocess, os, shutil

RESULTS_DIR = "/kaggle/working/evaluation"
REPO_DIR    = "/kaggle/working/motion-id"
REPO_RESULTS= os.path.join(REPO_DIR, "evaluation", "kaggle_results")
os.makedirs(REPO_RESULTS, exist_ok=True)

for fname in os.listdir(RESULTS_DIR):
    if fname.endswith(".csv") or fname.endswith(".md"):
        shutil.copy2(
            os.path.join(RESULTS_DIR, fname),
            os.path.join(REPO_RESULTS, fname))
        print(f"Copied: {fname}")

os.chdir(REPO_DIR)
subprocess.run(["git", "config", "user.email", "sreekar@motionid"], check=True)
subprocess.run(["git", "config", "user.name",  "Sreekar"],         check=True)
subprocess.run(["git", "add",    "evaluation/kaggle_results/"],    check=True)
subprocess.run(["git", "commit", "-m",
    "results: Kaggle P100 run"], check=True)
subprocess.run(["git", "push",   "origin", "main"],               check=True)
print("Results pushed to https://github.com/Sreekarji/motion-id")